In [14]:
import hashlib
import datetime
import os
import pytz
import re
import nltk
from nltk.tokenize import sent_tokenize

# Установка необходимых библиотек
%pip install python-docx -qqq
%pip install nltk -qqq
%pip install pypdf -qqq
%pip install striprtf -qqq # Установка striprtf для работы с RTF

# Загрузка токенизатора Punkt для русского языка (потребуется один раз)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# Загрузка специфического ресурса 'punkt_tab' для русской токенизации, если он требуется
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

from docx import Document
from pypdf import PdfReader
from striprtf.striprtf import rtf_to_text # Импорт для работы с RTF

def calculate_sha256(filepath):
    """Вычисляет SHA256 хэш файла."""
    sha256_hash = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except FileNotFoundError:
        return "Файл не найден"
    except Exception as e:
        return f"Ошибка при вычислении хэша: {e}"

def _has_multiple_sentences(text):
    """Проверяет, содержит ли текст более одного предложения, используя NLTK."""
    sentences = sent_tokenize(text, language='russian')
    meaningful_sentences = [s.strip() for s in sentences if len(s.strip()) > 5] # Увеличиваем минимальную длину для 'осмысленного' предложения
    return len(meaningful_sentences) >= 2

def get_annotation(filepath):
    """
    Извлекает аннотацию (первый абзац, который содержит два или более предложения)
    из документа. Поддерживаются файлы .docx, .txt, .pdf и .rtf.
    """
    file_extension = os.path.splitext(filepath)[1].lower()

    if file_extension == '.docx':
        try:
            doc = Document(filepath)
            for paragraph in doc.paragraphs:
                cleaned_text = paragraph.text.strip()
                if cleaned_text and _has_multiple_sentences(cleaned_text):
                    return cleaned_text
            return "В документе .docx не найдено абзацев с двумя или более предложениями."
        except Exception as e:
            return f"Ошибка при извлечении аннотации из .docx: {e}"
    elif file_extension == '.txt':
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                for line in f:
                    cleaned_line = line.strip()
                    if cleaned_line and _has_multiple_sentences(cleaned_line):
                        return cleaned_line
            return "В документе .txt не найдено строк с двумя или более предложениями."
        except Exception as e:
            return f"Ошибка при извлечении аннотации из .txt: {e}"
    elif file_extension == '.pdf':
        try:
            reader = PdfReader(filepath)
            # Собираем текст с нескольких первых страниц для поиска аннотации
            full_text = ""
            for i in range(min(len(reader.pages), 3)): # Проверяем первые 3 страницы
                full_text += reader.pages[i].extract_text() + "\n\n"

            # Разбиваем текст на "абзацы" (блоки текста, разделенные несколькими переносами строки)
            # и ищем первый подходящий
            text_blocks = full_text.split('\n\n')
            for block in text_blocks:
                cleaned_block = block.strip()
                if cleaned_block and _has_multiple_sentences(cleaned_block):
                    return cleaned_block
            return "В документе .pdf не найдено абзацев с двумя или более предложениями на первых страницах."
        except Exception as e:
            return f"Ошибка при извлечении аннотации из .pdf: {e}"
    elif file_extension == '.rtf':
        try:
            with open(filepath, 'r', encoding='latin-1') as f: # RTF often uses latin-1 or similar encodings
                rtf_content = f.read()
            plain_text = rtf_to_text(rtf_content)

            # Разбиваем текст на "абзацы" (блоки текста, разделенные несколькими переносами строки)
            text_blocks = plain_text.split('\n\n')
            for block in text_blocks:
                cleaned_block = block.strip()
                if cleaned_block and _has_multiple_sentences(cleaned_block):
                    return cleaned_block
            return "В документе .rtf не найдено абзацев с двумя или более предложениями."
        except Exception as e:
            return f"Ошибка при извлечении аннотации из .rtf: {e}"
    else:
        return f"Извлечение аннотации не поддерживается для файлов '{file_extension}'. В настоящее время поддерживаются только .docx, .txt, .pdf и .rtf."

# --- Основное выполнение ---

# Определите путь к вашему документу здесь. Пользователь может его изменить.
document_path = "/content/цифровые технологии в торговле.docx" # Пример: "/content/my_document.rtf"

# Проверяем, существует ли документ
if not os.path.exists(document_path):
    print(f"Ошибка: Документ не найден по пути: '{document_path}'")
else:
    document_hash = calculate_sha256(document_path)

    moscow_timezone = pytz.timezone('Europe/Moscow')
    calculation_datetime_moscow = datetime.datetime.now(moscow_timezone)
    calculation_date = calculation_datetime_moscow.strftime("%Y-%m-%d %H:%M:%S %Z%z")

    document_annotation = get_annotation(document_path)

    print(f"SHA256 Хэш: {document_hash}")
    print(f"Дата вычисления (МСК): {calculation_date}")
    print(f"Аннотация документа:")
    print(f"{document_annotation}")
    print(f"\n--- Дополнительная информация об аннотации ---")
    print(f"Длина аннотации: {len(document_annotation)} символов")
    print(f"Количество предложений в аннотации (по NLTK): {len(sent_tokenize(document_annotation, language='russian'))}")

    # Calculate total weight in bytes
    hash_bytes_length = len(document_hash) // 2 # SHA256 is 64 hex characters, representing 32 bytes
    annotation_bytes_length = len(document_annotation.encode('utf-8')) # Get byte length of the string encoded in UTF-8
    total_weight_bytes = hash_bytes_length + annotation_bytes_length

    print(f"Общий вес (хэш + аннотация): {total_weight_bytes} байт")


SHA256 Хэш: 657d12809fe0fcd5e408245b0e150052ed3b15f584ced8cd9daa79b0a3a1b27c
Дата вычисления (МСК): 2026-05-29 12:16:02 MSK+0300
Аннотация документа:
«Цифровизация» затронула все сферы человеческой жизни, но некоторые из них особенно. К таким «особенным» направлениям относится торговля товарами и услугами, которая интегрировала в себе ряд других отраслей (транспорт для производственной логистики и курьерской доставки, инфокоммуникации для систем заказов, доставки, управления производством, финансовые системы для взаиморасчетов покупателей, продавцов, курьеров и других участников технологического процесса.

--- Дополнительная информация об аннотации ---
Длина аннотации: 462 символов
Количество предложений в аннотации (по NLTK): 2
Общий вес (хэш + аннотация): 893 байт
